In [1]:
# =====================================================
# Version 3: Lasso Regression
# =====================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error


# =====================================================
# 1. 원본 데이터 불러오기
# =====================================================

train_raw = pd.read_csv('modulabs-di/train.csv')
test_raw = pd.read_csv('modulabs-di/test.csv')

y = train_raw['SalePrice']
test_ids = test_raw['Id']

X = train_raw.drop(columns=['SalePrice', 'Id'])
X_test = test_raw.drop(columns=['Id'])

In [2]:
# =====================================================
# 2. 새로운 변수 만들기
# =====================================================

def make_features(df):
    df = df.copy()

    df['TotalSF'] = (
        df['TotalBsmtSF'] +
        df['1stFlrSF'] +
        df['2ndFlrSF']
    )

    df['TotalBath'] = (
        df['FullBath'] +
        0.5 * df['HalfBath'] +
        df['BsmtFullBath'] +
        0.5 * df['BsmtHalfBath']
    )

    df['TotalPorchSF'] = (
        df['OpenPorchSF'] +
        df['EnclosedPorch'] +
        df['3SsnPorch'] +
        df['ScreenPorch']
    )

    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['RemodAge'] = df['YrSold'] - df['YearRemodAdd']

    df['QualityArea'] = (
        df['OverallQual'] * df['GrLivArea']
    )

    return df


X = make_features(X)
X_test = make_features(X_test)


In [3]:
# =====================================================
# 3. 결측치 처리
# =====================================================

none_cols = [
    'Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
    'BsmtFinType1', 'BsmtFinType2', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual',
    'GarageCond', 'PoolQC', 'Fence', 'MiscFeature',
    'MasVnrType'
]

zero_cols = [
    'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
    'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath',
    'GarageYrBlt', 'GarageCars', 'GarageArea',
    'MasVnrArea'
]

for col in none_cols:
    if col in X.columns:
        X[col] = X[col].fillna('None')
        X_test[col] = X_test[col].fillna('None')

for col in zero_cols:
    if col in X.columns:
        X[col] = X[col].fillna(0)
        X_test[col] = X_test[col].fillna(0)



In [4]:
# =====================================================
# 4. 수치형 / 범주형 컬럼 구분
# =====================================================

categorical_cols = X.select_dtypes(
    include=['object', 'category']
).columns.tolist()

numeric_cols = X.select_dtypes(
    include='number'
).columns.tolist()


# =====================================================
# 5. 전처리 파이프라인
# =====================================================

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(
        handle_unknown='ignore'
    ))
])

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer([
    ('cat', categorical_pipeline, categorical_cols),
    ('num', numeric_pipeline, numeric_cols)
])


C:\Users\chaee\AppData\Local\Temp\ipykernel_10024\3156487114.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(


In [5]:
# =====================================================
# 6. 학습 / 검증 데이터 분리
# =====================================================

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# 집값 로그 변환
y_train_log = np.log1p(y_train)


In [6]:
# =====================================================
# 7. 여러 alpha 실험
# =====================================================

alphas = [
    0.0001,
    0.0003,
    0.0005,
    0.001,
    0.003,
    0.01,
    0.03
]

results = []

best_rmse = float('inf')
best_alpha = None
best_model = None

for alpha in alphas:

    model = Pipeline([
        ('preprocessor', preprocessor),

        ('regressor', Lasso(
            alpha=alpha,
            max_iter=50000,
            tol=0.0001
        ))
    ])

    model.fit(X_train, y_train_log)

    # 검증 데이터 예측
    valid_pred_log = model.predict(X_valid)

    # 실제 집값으로 복원
    valid_pred = np.expm1(valid_pred_log)
    valid_pred = np.maximum(valid_pred, 0)

    rmse = np.sqrt(
        mean_squared_error(y_valid, valid_pred)
    )

    results.append({
        'alpha': alpha,
        'RMSE': rmse
    })

    print(f"alpha={alpha:.4f} | RMSE={rmse:.2f}")

    if rmse < best_rmse:
        best_rmse = rmse
        best_alpha = alpha
        best_model = model



alpha=0.0001 | RMSE=19679.73
alpha=0.0003 | RMSE=19610.33
alpha=0.0005 | RMSE=19897.44
alpha=0.0010 | RMSE=21549.06
alpha=0.0030 | RMSE=24227.52
alpha=0.0100 | RMSE=25764.13
alpha=0.0300 | RMSE=28825.38


In [7]:
# =====================================================
# 8. 가장 좋은 alpha 확인
# =====================================================

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('RMSE')

print("\n===== Alpha 실험 결과 =====")
print(results_df.to_string(index=False))

print("\nBest Alpha:", best_alpha)
print("Best Validation RMSE:", round(best_rmse, 2))




===== Alpha 실험 결과 =====
 alpha         RMSE
0.0003 19610.327661
0.0001 19679.729842
0.0005 19897.443259
0.0010 21549.061478
0.0030 24227.521367
0.0100 25764.126467
0.0300 28825.379566

Best Alpha: 0.0003
Best Validation RMSE: 19610.33


In [8]:
# =====================================================
# 9. 전체 학습 데이터로 재학습
# =====================================================

best_model.fit(X, np.log1p(y))


# =====================================================
# 10. 테스트 데이터 예측
# =====================================================

test_pred_log = best_model.predict(X_test)

test_pred = np.expm1(test_pred_log)
test_pred = np.maximum(test_pred, 0)

In [9]:
# =====================================================
# 11. 제출 파일 생성
# =====================================================

submission_v3 = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': test_pred
})

submission_v3.to_csv(
    'submission_v3_lasso.csv',
    index=False
)

print("\n제출 파일 생성 완료!")
print(submission_v3.head())
print("파일 크기:", submission_v3.shape)



제출 파일 생성 완료!
    Id      SalePrice
0   86  255222.675037
1  227  264103.861294
2  492  144588.416608
3   64  120519.982948
4  576  114401.962625
파일 크기: (292, 2)
